# Normative Model

Train a quantile regression MLP that predicts the 10th, 50th and 90th percentiles of Functional Connectivity (FC) given Age and Sex.

Input data: `../data/processed/abide_normative_cleaned.csv` (from `preprocessing.ipynb`)

In [1]:
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
import os
import json
import random

In [2]:
# Random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device: GPU (cuda), Apple Silicon (mps) or cpu
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

Device: mps
PyTorch version: 2.5.1


## Load Data

Loading the preprocessed dataset and the scaling parameters (used later to transform predictions back to FC units).

In [3]:
# Preprocessed dataset
df = pd.read_csv("../data/processed/abide_normative_cleaned.csv")
print(f"Total samples: {len(df)}")

# Scaled parameters
with open("../data/processed/scaling_params.json", "r") as f:
    scaling_params = json.load(f)

age_mean = scaling_params['age_mean']
age_std = scaling_params['age_std']
fc_mean = scaling_params['fc_mean']
fc_std = scaling_params['fc_std']
print(scaling_params)

df.head()

Total samples: 468
{'age_mean': 16.83903311965812, 'age_std': 7.242265848499033, 'fc_mean': 0.5217374600625769, 'fc_std': 0.1984266098918636}


,AGE_AT_SCAN,SEX,FC,Age_scaled,FC_scaled
0,25.12,0,0.612840,1.143422,0.459125
1,12.92,0,0.435302,-0.541134,-0.435606
2,19.80,0,0.537265,0.408845,0.078256
3,12.15,0,0.709478,-0.647454,0.946148
4,14.77,0,0.759157,-0.285689,1.196512


**Dataset and DataLoader**

Converting the inputs (`Age_scaled`, `SEX`) and target (`FC_scaled`) into a PyTorch `TensorDataset`, and splitting the data into training (70%), validation (15%) and testing (15%) sets.

In [4]:
# Inputs and target as tensors
X = torch.tensor(df[['Age_scaled', 'SEX']].values, dtype=torch.float32)
y = torch.tensor(df['FC_scaled'].values, dtype=torch.float32).unsqueeze(1)

# call dataset
dataset = TensorDataset(X, y)

# Split data (70/15/15)
generator = torch.Generator().manual_seed(SEED)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size], generator=generator)

print(f"Training set size: {len(train_dataset)} samples")
print(f"Validation set size: {len(val_dataset)} samples")
print(f"Test set size: {len(test_dataset)} samples")

# dataloaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Training set size: 327 samples
Validation set size: 70 samples
Test set size: 71 samples
